# 00_workload_distributor_pipeline
Baseline pipeline (workload -> time-series -> simple thermal/water model -> predictor)

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

base_dir = r"C:\Users\RANADEEP\Documents\IBM Project\Alibaba_v2018"
raw_dir = os.path.join(base_dir, "01_raw_data")
processed_dir = os.path.join(base_dir, "02_processed_data")
weather_dir = os.path.join(base_dir, "03_weather_data")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(weather_dir, exist_ok=True)


In [2]:
# Load workload data with correct column handling
print('Loading machine_usage.csv...')
df_machine = pd.read_csv(
    os.path.join(raw_dir, 'machine_usage.csv'),
    header=None,
    usecols=[0, 1, 2],
    names=['machine_id', 'timestamp', 'cpu_util_percent']
)

df_machine['timestamp'] = pd.to_numeric(df_machine['timestamp'], errors='coerce')
df_machine['cpu_util_percent'] = pd.to_numeric(df_machine['cpu_util_percent'], errors='coerce').fillna(0)

# Aggregate to time-series (hourly)
ts_df = df_machine.groupby('timestamp')['cpu_util_percent'].mean().reset_index()
ts_df = ts_df.sort_values('timestamp')

# Add datetime for later merging
ts_df['datetime'] = pd.to_datetime('2018-01-01') + pd.to_timedelta(ts_df['timestamp'], unit='s')

print('Time-series shape:', ts_df.shape)


Loading machine_usage.csv...
Time-series shape: (52748, 3)


In [3]:
# Simple thermal/water model
P_idle = 150  # W
P_max = 400   # W
ts_df['power_kw'] = (P_idle + (P_max - P_idle) * (ts_df['cpu_util_percent'] / 100.0)) / 1000.0

WUE = 1.5  # liters per kWh
ts_df['water_liters'] = ts_df['power_kw'] * WUE

# Save baseline dataset
ts_df.to_csv(os.path.join(processed_dir, 'workload_timeseries.csv'), index=False)
print('Saved to processed/workload_timeseries.csv')


Saved to processed/workload_timeseries.csv


In [4]:
# Predictor with CORRECT train_test_split usage
X = ts_df[['cpu_util_percent']]
y_power = ts_df['power_kw']
y_water = ts_df['water_liters']

# Split once, then reuse indices for both targets
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_power_train, y_power_test = y_power.iloc[train_idx], y_power.iloc[test_idx]
y_water_train, y_water_test = y_water.iloc[train_idx], y_water.iloc[test_idx]

model_power = RandomForestRegressor(n_estimators=50, random_state=42)
model_power.fit(X_train, y_power_train)
pred_power = model_power.predict(X_test)

model_water = RandomForestRegressor(n_estimators=50, random_state=42)
model_water.fit(X_train, y_water_train)
pred_water = model_water.predict(X_test)

print(f"Power MAE: {mean_absolute_error(y_power_test, pred_power):.4f} kW")
print(f"Water MAE: {mean_absolute_error(y_water_test, pred_water):.4f} Liters")


Power MAE: 0.0000 kW
Water MAE: 0.0000 Liters
